# Transformer v3 low-memory predict

按月份分块读取和推理，避免评分系统 OOM。


In [2]:
"""
Transformer v3 low-memory online inference notebook for BigQuant.

Upload together with transformer_train_v3.py and transformer_model_v3.json.
This notebook never trains in main(); it loads JSON weights and predicts by
calendar chunks to avoid BigDB / pandas OOM in the official scorer.
"""
import os
import gc

import numpy as np
import pandas as pd
import torch

try:
    import dai  # type: ignore
except Exception:
    dai = None

try:
    import structlog  # type: ignore
    logger = structlog.get_logger()
except Exception:
    import logging
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)

from transformer_train_v3 import (
    MODEL_PATH,
    StockTransformer,
    load_model,
    pool,
    predict_in_chunks,
    query_official_pool,
    select_table,
)


def _load_inference_model(model_path=MODEL_PATH):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = load_model(model_path, map_location=device)
    model = StockTransformer(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    stats = (
        np.asarray(ckpt["mean"], dtype=np.float32),
        np.asarray(ckpt["std"], dtype=np.float32),
    )
    return model, stats, device


def main(datasources, start_date, end_date):
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到模型文件 {MODEL_PATH}; 请上传 transformer_model_v3.json，不要在评分阶段训练。"
        )

    table = select_table(datasources)
    model, stats, device = _load_inference_model(MODEL_PATH)
    logger.info("model loaded", path=MODEL_PATH, table=table, device=str(device))

    try:
        instruments = pool(start_date, end_date)
    except Exception:
        instruments = None

    idx_df = predict_in_chunks(
        model=model,
        table=table,
        start_date=start_date,
        end_date=end_date,
        stats=stats,
        device=device,
        instruments=instruments,
    )

    if dai is not None:
        try:
            stk = query_official_pool(start_date, end_date)
            idx_df["date"] = pd.to_datetime(idx_df["date"]).dt.normalize()
            idx_df = pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
        except Exception:
            pass

    result = (
        idx_df.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )
    del idx_df
    gc.collect()
    logger.info("score data ready", rows=len(result))
    return result


if __name__ == "__main__":
    datasources = {"bar30m": "bigalpha_2026_stock_bar30m"}
    start_date, end_date = "2019-01-01 00:00:00", "2024-12-31 23:59:59"
    print(main(datasources, start_date, end_date).head())


[2026-07-31 10:38:03] [info     ] model loaded                   device=cuda path=/home/aiuser/work/AI因子比赛/提交5/transformer_model_v3.json table=bigalpha_2026_stock_bar30m
[2026-07-31 10:38:05] [info     ] infer chunk built              elapsed=1.58 end='2019-01-31 23:59:59' samples=24237 start=2019-01-01
[2026-07-31 10:38:07] [info     ] infer chunk built              elapsed=1.81 end='2019-02-28 23:59:59' samples=24287 start=2019-02-01
[2026-07-31 10:38:09] [info     ] infer chunk built              elapsed=2.02 end='2019-03-31 23:59:59' samples=34000 start=2019-03-01
[2026-07-31 10:38:12] [info     ] infer chunk built              elapsed=2.15 end='2019-04-30 23:59:59' samples=34139 start=2019-04-01
[2026-07-31 10:38:14] [info     ] infer chunk built              elapsed=2.11 end='2019-05-31 23:59:59' samples=32717 start=2019-05-01
[2026-07-31 10:38:17] [info     ] infer chunk built              elapsed=1.95 end='2019-06-30 23:59:59' samples=31198 start=2019-06-01
[2026-07-31 10:38:19